
# SOLID Principles in Python

This notebook demonstrates the **SOLID** design principles using Object-Oriented Programming (OOP) in Python. 
We will look at "Wrong" (violating the principle) and "Right" (adhering to the principle) implementations for each.

**SOLID** stands for:
- **S** - Single Responsibility Principle (SRP)
- **O** - Open/Closed Principle (OCP)
- **L** - Liskov Substitution Principle (LSP)
- **I** - Interface Segregation Principle (ISP)
- **D** - Dependency Inversion Principle (DIP)



## 1. Single Responsibility Principle (SRP)

**Definition:** A class should have only one reason to change, meaning it should have only one job or responsibility.

### The Problem (Wrong Implementation)
Here, the `UserManager` class handles user creation, email validation, *and* sending welcome emails. It has too many responsibilities.


In [ ]:

# ❌ WRONG: Violating SRP

class UserManager:
    def create_user(self, username, email):
        # Responsibility 1: User Management
        self.username = username
        self.email = email
        
        # Responsibility 2: Validation
        if '@' not in email:
            raise ValueError("Invalid email")
            
        # Responsibility 3: Notification
        print(f"Sending welcome email to {email}...")

# Usage
try:
    user_mgr = UserManager()
    user_mgr.create_user("john_doe", "john@example.com")
except ValueError as e:
    print(e)



### The Solution (Right Implementation)
We split the responsibilities into separate classes: `User` (data), `EmailValidator` (validation), and `EmailService` (notification).


In [ ]:

# ✅ RIGHT: Adhering to SRP

class EmailValidator:
    @staticmethod
    def validate(email):
        if '@' not in email:
            raise ValueError("Invalid email")

class EmailService:
    def send_welcome_email(self, email):
        print(f"Sending welcome email to {email}...")

class UserManager:
    def __init__(self):
        self.email_service = EmailService()

    def create_user(self, username, email):
        # Delegate validation
        EmailValidator.validate(email)
        
        # Core logic: Create user (simplified)
        user = {"username": username, "email": email}
        
        # Delegate notification
        self.email_service.send_welcome_email(email)
        return user

# Usage
try:
    user_mgr = UserManager()
    user_mgr.create_user("john_doe", "john@example.com")
except ValueError as e:
    print(e)



## 2. Open/Closed Principle (OCP)

**Definition:** Software entities (classes, modules, functions, etc.) should be open for extension, but closed for modification.

### The Problem (Wrong Implementation)
The `DiscountCalculator` has to be modified every time we add a new customer type (e.g., if we adding "Platinum").


In [ ]:

# ❌ WRONG: Violating OCP

class DiscountCalculator:
    def calculate_discount(self, customer_type, amount):
        if customer_type == "Regular":
            return amount * 0.1
        elif customer_type == "Premium":
            return amount * 0.2
        # If we want to add "VIP", we must modify this class directly
        elif customer_type == "VIP":
            return amount * 0.3
        else:
            return 0
            
# Usage
calc = DiscountCalculator()
print(f"Regular Discount: {calc.calculate_discount('Regular', 100)}")
print(f"Premium Discount: {calc.calculate_discount('Premium', 100)}")



### The Solution (Right Implementation)
We use an abstract base class (or interface) for the discount strategy. New customer types can be added by creating new classes without modifying existing code.


In [ ]:

# ✅ RIGHT: Adhering to OCP

from abc import ABC, abstractmethod

class DiscountStrategy(ABC):
    @abstractmethod
    def get_discount(self, amount):
        pass

class RegularDiscount(DiscountStrategy):
    def get_discount(self, amount):
        return amount * 0.1

class PremiumDiscount(DiscountStrategy):
    def get_discount(self, amount):
        return amount * 0.2

class VIPDiscount(DiscountStrategy):
    def get_discount(self, amount):
        return amount * 0.3

class DiscountCalculator:
    def calculate(self, strategy: DiscountStrategy, amount):
        return strategy.get_discount(amount)

# Usage
calc = DiscountCalculator()
print(f"Regular Discount: {calc.calculate(RegularDiscount(), 100)}")
print(f"Premium Discount: {calc.calculate(PremiumDiscount(), 100)}")



## 3. Liskov Substitution Principle (LSP)

**Definition:** Objects of a superclass shall be replaceable with objects of its subclasses without breaking the application.

### The Problem (Wrong Implementation)
A `Penguin` is a `Bird`, but it cannot fly. If we substitute `Bird` with `Penguin`, the generic `let_bird_fly` function crashes.


In [ ]:

# ❌ WRONG: Violating LSP

class Bird:
    def fly(self):
        return "I can fly!"

class Sparrow(Bird):
    pass

class Penguin(Bird):
    def fly(self):
        # Penguins can't fly, so this breaks the expectation of the base class
        raise Exception("I cannot fly")

def let_bird_fly(bird: Bird):
    print(bird.fly())

# Usage
try:
    sparrow = Sparrow()
    let_bird_fly(sparrow) # Works
    
    penguin = Penguin()
    let_bird_fly(penguin) # Crashes!
except Exception as e:
    print(f"Error: {e}")



### The Solution (Right Implementation)
We separate the hierarchy. `Bird` is a general concept, while `FlyingBird` adds the flight capability. `Penguin` inherits from `Bird` but not `FlyingBird`.


In [ ]:

# ✅ RIGHT: Adhering to LSP

class Bird:
    def eat(self):
        return "I am eating"

class FlyingBird(Bird):
    def fly(self):
        return "I can fly!"

class Sparrow(FlyingBird):
    pass

class Penguin(Bird):
    def swim(self):
        return "I can swim!"

def let_bird_fly(bird: FlyingBird):
    print(bird.fly())

# Usage
sparrow = Sparrow()
let_bird_fly(sparrow) # Works

penguin = Penguin()
# let_bird_fly(penguin)  # This would fail static type checking or be obviously wrong usage, 
                         # but we don't force Penguin to implement a broken 'fly' method.
print(penguin.eat())
print(penguin.swim())



## 4. Interface Segregation Principle (ISP)

**Definition:** Clients should not be forced to depend upon interfaces that they do not use.

### The Problem (Wrong Implementation)
A `MultiFunctionDevice` abstract class forces all subclasses to implement print, scan, and fax. A `SimplePrinter` doesn't need scan or fax.


In [ ]:

# ❌ WRONG: Violating ISP

from abc import ABC, abstractmethod

class MultiFunctionDevice(ABC):
    @abstractmethod
    def print_doc(self, doc):
        pass
        
    @abstractmethod
    def scan_doc(self, doc):
        pass
        
    @abstractmethod
    def fax_doc(self, doc):
        pass

class SimplePrinter(MultiFunctionDevice):
    def print_doc(self, doc):
        print(f"Printing {doc}")
        
    def scan_doc(self, doc):
        # Force implementation of unnecessary method
        raise NotImplementedError("Not supported")
        
    def fax_doc(self, doc):
        # Force implementation of unnecessary method
        raise NotImplementedError("Not supported")

# Usage
printer = SimplePrinter()
printer.print_doc("Resume.pdf")
try:
    printer.scan_doc("Resume.pdf")
except Exception as e:
    print(e)



### The Solution (Right Implementation)
We segregate the large interface into smaller, specific interfaces: `Printer`, `Scanner`, `Fax`.


In [ ]:

# ✅ RIGHT: Adhering to ISP

from abc import ABC, abstractmethod

class Printer(ABC):
    @abstractmethod
    def print_doc(self, doc):
        pass

class Scanner(ABC):
    @abstractmethod
    def scan_doc(self, doc):
        pass

class Fax(ABC):
    @abstractmethod
    def fax_doc(self, doc):
        pass

class SimplePrinter(Printer):
    def print_doc(self, doc):
        print(f"Printing {doc}")

class Photocopier(Printer, Scanner):
    def print_doc(self, doc):
        print(f"Printing {doc}")
        
    def scan_doc(self, doc):
        print(f"Scanning {doc}")

# Usage
simple = SimplePrinter()
simple.print_doc("Resume.pdf")
# simple.scan_doc("Resume.pdf") # Method doesn't exist, which is correct.

office_machine = Photocopier()
office_machine.print_doc("Contract.pdf")
office_machine.scan_doc("Contract.pdf")



## 5. Dependency Inversion Principle (DIP)

**Definition:** High-level modules should not depend on low-level modules. Both should depend on abstractions.

### The Problem (Wrong Implementation)
The `Switch` class (high-level) creates a direct instance of `LightBulb` (low-level). If we want to switch on a `Fan`, we have to modify the `Switch` class.


In [ ]:

# ❌ WRONG: Violating DIP

class LightBulb:
    def turn_on(self):
        print("LightBulb: On")
        
    def turn_off(self):
        print("LightBulb: Off")

class ElectricPowerSwitch:
    def __init__(self, bulb: LightBulb):
        # Direct dependency on concrete class
        self.bulb = bulb
        self.on = False
        
    def press(self):
        if self.on:
            self.bulb.turn_off()
            self.on = False
        else:
            self.bulb.turn_on()
            self.on = True

# Usage
bulb = LightBulb()
switch = ElectricPowerSwitch(bulb)
switch.press()



### The Solution (Right Implementation)
The `Switch` depends on an abstraction `Switchable`. Both `LightBulb` and `Fan` implement this abstraction.


In [ ]:

# ✅ RIGHT: Adhering to DIP

from abc import ABC, abstractmethod

# Abstraction
class Switchable(ABC):
    @abstractmethod
    def turn_on(self):
        pass
        
    @abstractmethod
    def turn_off(self):
        pass

# Low-level module 1
class LightBulb(Switchable):
    def turn_on(self):
        print("LightBulb: On")
        
    def turn_off(self):
        print("LightBulb: Off")

# Low-level module 2
class Fan(Switchable):
    def turn_on(self):
        print("Fan: On")
        
    def turn_off(self):
        print("Fan: Off")

# High-level module
class ElectricPowerSwitch:
    def __init__(self, client: Switchable):
        self.client = client
        self.on = False
        
    def press(self):
        if self.on:
            self.client.turn_off()
            self.on = False
        else:
            self.client.turn_on()
            self.on = True

# Usage
bulb = LightBulb()
switch_1 = ElectricPowerSwitch(bulb)
switch_1.press()

fan = Fan()
switch_2 = ElectricPowerSwitch(fan)
switch_2.press()
